In [1]:
import sys

sys.path.append("/kaggle/input/models/minhkhanhdoan/deepland-mid-224/pytorch/default/1")

In [2]:
import json
import os
import random
import time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.amp import autocast, GradScaler

from datasets_224 import get_dataloaders
from model_defs import NetMid
from train_config import MID_CONFIG
from train_utils import save_history_csv, save_plots


def set_seed(seed=42):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def run_one_epoch(model, loader, criterion, optimizer, device, scaler, is_train):
    model.train() if is_train else model.eval()

    total_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad(set_to_none=True)
            with autocast("cuda"):
                outputs = model(images)
                loss = criterion(outputs, labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            with torch.no_grad():
                with autocast("cuda"):
                    outputs = model(images)
                    loss = criterion(outputs, labels)

        total_loss += loss.item() * images.size(0)
        _, preds = outputs.max(1)
        correct += preds.eq(labels).sum().item()
        total += labels.size(0)

    return total_loss / total, 100.0 * correct / total


def train():
    cfg = MID_CONFIG

    data_root = "/kaggle/input/datasets/minhkhanhdoan/dataset-deepland/datasets"  # sửa lại
    output_root = "/kaggle/working"

    set_seed(cfg["seed"])
    os.makedirs(output_root, exist_ok=True)

    device = torch.device("cuda")
    torch.backends.cudnn.benchmark = True

    print("Loading data...")
    train_loader, test_loader, class_names = get_dataloaders(
        data_root=data_root,
        batch_size=16,
        num_workers=4,
        seed=cfg["seed"],
        print_summary=False,
    )

    print("Data ready")

    n_class = len(class_names)
    model = NetMid(image_size=224, n_class=n_class).to(device)
    model = torch.compile(model)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(model.parameters(), lr=cfg["learning_rate"], weight_decay=cfg["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg["epochs"])
    scaler = GradScaler("cuda")

    best_acc = 0.0
    best_epoch = 0

    for epoch in range(1, cfg["epochs"] + 1):
        t0 = time.time()

        train_loss, train_acc = run_one_epoch(model, train_loader, criterion, optimizer, device, scaler, True)
        test_loss, test_acc = run_one_epoch(model, test_loader, criterion, optimizer, device, scaler, False)
        scheduler.step()

        print(f"{epoch}/{cfg['epochs']} | {train_acc:.1f}% | {test_acc:.1f}% | {time.time()-t0:.1f}s")

        if test_acc > best_acc:
            best_acc = test_acc
            best_epoch = epoch
            torch.save(model.state_dict(), f"{output_root}/best.pth")

    print("BEST:", best_acc, best_epoch)


train()

Loading data...
Data ready


W0406 16:44:44.378000 25 torch/_inductor/utils.py:1679] [0/0] Not enough SMs to use max_autotune_gemm mode


1/100 | 20.3% | 20.0% | 166.4s
2/100 | 21.1% | 19.6% | 34.4s
3/100 | 34.8% | 26.4% | 15.0s
4/100 | 43.0% | 32.0% | 15.2s
5/100 | 48.4% | 40.8% | 15.1s
6/100 | 51.8% | 38.8% | 15.4s
7/100 | 56.7% | 36.0% | 15.7s
8/100 | 57.9% | 38.8% | 15.6s
9/100 | 60.8% | 42.0% | 15.5s
10/100 | 63.2% | 41.6% | 15.4s
11/100 | 64.4% | 36.0% | 15.4s
12/100 | 67.0% | 43.2% | 15.4s
13/100 | 71.3% | 37.6% | 15.5s
14/100 | 72.2% | 41.2% | 15.4s
15/100 | 74.8% | 45.6% | 15.3s
16/100 | 80.4% | 39.6% | 15.4s
17/100 | 81.3% | 46.0% | 15.5s
18/100 | 81.9% | 43.6% | 15.4s
19/100 | 84.3% | 37.6% | 15.5s
20/100 | 87.2% | 38.8% | 15.4s
21/100 | 88.7% | 43.6% | 15.4s
22/100 | 89.3% | 40.0% | 15.5s
23/100 | 92.2% | 42.4% | 15.4s
24/100 | 92.2% | 39.6% | 15.4s
25/100 | 94.2% | 44.4% | 15.5s
26/100 | 95.2% | 38.8% | 15.2s
27/100 | 93.6% | 40.0% | 15.3s
28/100 | 96.4% | 42.8% | 15.3s
29/100 | 97.0% | 35.2% | 15.4s
30/100 | 96.2% | 45.6% | 15.3s
31/100 | 96.2% | 43.2% | 15.3s
32/100 | 98.2% | 44.4% | 15.3s
33/100 | 98.0% |

In [3]:
import matplotlib.pyplot as plt

def plot_history(history):
    epochs = range(1, len(history["train_loss"]) + 1)

    plt.figure()
    plt.plot(epochs, history["train_loss"], label="Train Loss")
    plt.plot(epochs, history["test_loss"], label="Test Loss")
    plt.legend()
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.show()

    plt.figure()
    plt.plot(epochs, history["train_acc"], label="Train Acc")
    plt.plot(epochs, history["test_acc"], label="Test Acc")
    plt.legend()
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.show()